## Hands-On Lab 5 - Tuning And Testing

In this lab, you will experiment with differencing and tuning the random forest model from the previous lab. Then, you will perform final testing with the tuned model to determine whether it can be useful.

### Step 1 - Load the Training Data

Run the following code to load the Bike Sharing training data and set the index.

In [ ]:
import pandas as pd

# Load training data
bike_sharing_train = pd.read_csv('BikeSharing_Train.csv', parse_dates = ['date_time'])

# Set index
bike_sharing_train = bike_sharing_train.set_index('date_time')
bike_sharing_train = bike_sharing_train.asfreq('h')
bike_sharing_train = bike_sharing_train.sort_index()

### Step 2 - Load the Validation Data

Run the following code to load the Bike Sharing validation data and set the index.

In [ ]:
# Load validation data
bike_sharing_val = pd.read_csv('BikeSharing_Validation.csv', parse_dates = ['date_time'])

# Set index
bike_sharing_val = bike_sharing_val.set_index('date_time')
bike_sharing_val = bike_sharing_val.asfreq('h')
bike_sharing_val = bike_sharing_val.sort_index()

### Step 3 - Chart Set Up

Run the following code to set up line charts using dark mode.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from skforecast.plot import set_dark_theme

# Set fonts/grid defaults
custom_style = {
    'axes.titlesize': 18,
    'axes.labelsize': 15,
    'xtick.labelsize': 13,
    'ytick.labelsize': 13,
    'grid.alpha': 0.3,
}

# Make dark mode the default for all plots
set_dark_theme(custom_style = custom_style) 

### Step 4 - Forecast KPI Calculations

Run the following code for the utility function that calculates forecasting model KPIs (i.e., bias and MAE).

In [ ]:
import numpy as np

# Utility function for our model KPIs
def forecast_kpis(forecasts, targets):
    # Bias calculations
    errors = forecasts - targets 
    targets_sum = np.sum(targets)
    bias = np.mean(errors)
    bias_pct = np.sum(errors) / targets_sum 

    # MAE calculations
    mae = np.mean(np.abs(errors))
    mae_pct = np.sum(np.abs(errors)) / targets_sum

    # Display
    print(f'Bias: {bias:0.4f}, {(bias_pct * 100):0.2f}%')
    print(f'MAE: {mae:0.4f}, {(mae_pct * 100):0.2f}%') 

### Step 5 - Lags & Endogenous Features

Run the following code to create the lags and rolling window features.

In [ ]:
from skforecast.preprocessing import RollingFeatures

# Forecast horizon
h = 168

# Create a list of the "optimal" lags from PACF
pacf_lags = [1, 2, 5, 10, 15, 18, 22, 25]

# Create endogenous rolling window features
window_features = RollingFeatures(stats = 'max', window_sizes = 5)

### Step 6 - Exogenous Features

Run the following code to create the exogenous features.

In [ ]:
# Create a list of categorical features to one-hot encode
categorical_features = ['weekday']

# One-hot encode the features and create a new DataFrame for training
bike_sharing_exog_train = pd.get_dummies(bike_sharing_train, columns = categorical_features, 
                                         prefix = categorical_features, dtype = int)

# One-hot encode the features and create a new DataFrame for validation
bike_sharing_exog_val = pd.get_dummies(bike_sharing_val, columns = categorical_features, 
                                       prefix = categorical_features, dtype = int)

# Create a list of exogenous features
exog_features = ['holiday','workingday',
                 'weekday_0', 'weekday_1', 'weekday_2', 'weekday_3', 'weekday_4', 
                 'weekday_5', 'weekday_6']

### Step 7 - Train Reference Model

Run the following code to train the random forest model from the last lab.

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from skforecast.recursive import ForecasterRecursive

# Forecast using a Random Forest "optimal" lags & rolling features
rf_lags_roll_exog = ForecasterRecursive(regressor = RandomForestRegressor(random_state = 12345), 
                                        lags = pacf_lags, window_features = window_features)

# Learn from the lagged & rolling dataset
rf_lags_roll_exog.fit(y = bike_sharing_train['users'],
                      exog = bike_sharing_exog_train[exog_features])

# Make predictions to compare against validation set
rf_lags_roll_exog_preds = rf_lags_roll_exog.predict(steps = h,
                                                    exog = bike_sharing_exog_val[exog_features])

# What is the accuracy on the validation set?
forecast_kpis(rf_lags_roll_exog_preds, bike_sharing_val['users'].head(n = h))

### Step 8 - Differencing

As you saw in the lecture, differencing enables decision tree-based machine learning models (e.g., random forests) to extrapolate forecasts beyond the training target values. Enter and run the following code to see if differencing might be useful for the Bike Sharing dataset.

In [ ]:
# Enter your lab code here

### Step 9 - Tuning

It's often the case that increasing the number of trees in a random forest can improve forecast KPI performance. Enter and run the following code that increases the *n_estimators* hyperparameter from the default of 100 trees to 200 trees.

In [ ]:
# Enter your lab code here

### Step 10 - Load the Test Data

Run the following code to load the Bike Sharing test data and set the index.

In [ ]:
# Load test data
bike_sharing_test = pd.read_csv('BikeSharing_Test.csv', parse_dates = ['date_time'])

# Set index
bike_sharing_test = bike_sharing_test.set_index('date_time')
bike_sharing_test = bike_sharing_test.asfreq('h')
bike_sharing_test = bike_sharing_test.sort_index()

### Step 11 - Exogenous Features (Test)

Any exogenous features that were used to train the ML forecasting model must be present in the test dataset. Run the following code to create the exongenous features.

In [ ]:
# One-hot encode the features and create a new DataFrame for testing
bike_sharing_exog_test = pd.get_dummies(bike_sharing_test, columns = categorical_features, 
                                        prefix = categorical_features, dtype = int)

# Get info
bike_sharing_exog_test.info()

### Step 12 - Final Testing

Final testing consists of the following steps for this lab:

1. Combining the training and validation datasets.
2. Training a tuned random forest model with endogenous and exogenous features.
3. Making predictions for the test dataset.
4. Calculating the KPIs for the test forecasts.

Run the following code to perform the final testing.

In [ ]:
# Concatenate train and validation datasets for final testing
bike_sharing_final_train = pd.concat([bike_sharing_exog_train, bike_sharing_exog_val])

# Train the final RF model using everything learned so far
rf_final = ForecasterRecursive(regressor = RandomForestRegressor(random_state = 12345,
                                                                 n_estimators = 200), 
                               lags = pacf_lags, window_features = window_features)

# Learn from the lagged dataset
rf_final.fit(y = bike_sharing_final_train['users'], 
             exog = bike_sharing_final_train[exog_features],
             store_in_sample_residuals = True)

# Make predictions to compare against test set
rf_final_preds = rf_final.predict(steps = h, exog = bike_sharing_exog_test[exog_features])

# What is the accuracy on the test set?
forecast_kpis(rf_final_preds, bike_sharing_exog_test['users'].head(n = h))

### Step 13 - Visualize the Final Model

Run the following code to visualize the final random forest model's forecasts versus the test dataset observations.

In [ ]:
# Create a plot 11 inches wide and 5 inches high
fig, ax = plt.subplots(figsize = (11, 5))

# Validation targets
bike_sharing_test['users'].head(n = h).plot(ax = ax, linewidth = 2)

# Forecasts
rf_final_preds.plot(ax = ax, linewidth = 2, linestyle = '--', 
                    label = 'Final Model Predictions')

# Display plot
ax.set_title('Final Model Predictions for Test Dataset')
ax.legend()
fig.tight_layout()
plt.show()